In [ ]:
from google.colab import files
files.upload()

In [ ]:
!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/a
!chmod 600 ~/.kaggle/kaggle.json

In [7]:
!kaggle datasets download -d orvile/brain-cancer-mri-dataset
!unzip brain-cancer-mri-dataset.zip -d /content/

Streaming output truncated to the last 5000 lines.
  inflating: /content/Brain_Cancer raw MRI data/Brain_Cancer/brain_glioma/brain_glioma_1058.jpg  
  inflating: /content/Brain_Cancer raw MRI data/Brain_Cancer/brain_glioma/brain_glioma_1059.jpg  
  inflating: /content/Brain_Cancer raw MRI data/Brain_Cancer/brain_glioma/brain_glioma_1060.jpg  
  inflating: /content/Brain_Cancer raw MRI data/Brain_Cancer/brain_glioma/brain_glioma_1061.jpg  
  inflating: /content/Brain_Cancer raw MRI data/Brain_Cancer/brain_glioma/brain_glioma_1062.jpg  
  inflating: /content/Brain_Cancer raw MRI data/Brain_Cancer/brain_glioma/brain_glioma_1063.jpg  
  inflating: /content/Brain_Cancer raw MRI data/Brain_Cancer/brain_glioma/brain_glioma_1064.jpg  
  inflating: /content/Brain_Cancer raw MRI data/Brain_Cancer/brain_glioma/brain_glioma_1065.jpg  
  inflating: /content/Brain_Cancer raw MRI data/Brain_Cancer/brain_glioma/brain_glioma_1066.jpg  
  inflating: /content/Brain_Cancer raw MRI data/Brain_Cancer/brain_

In [8]:
import os

print(os.listdir("/content"))

print(os.listdir("/content/Brain_Cancer raw MRI data"))

print(os.listdir("/content/Brain_Cancer raw MRI data/Brain_Cancer"))

dataset = "/content/Brain_Cancer raw MRI data/Brain_Cancer"
print("✅ Dataset Path Set To:", dataset)

['.config', 'Brain_Cancer raw MRI data', 'brain-cancer-mri-dataset.zip', 'dataset.csv', 'sample_data']
['Brain_Cancer']
['brain_tumor', 'brain_menin', 'brain_glioma']
✅ Dataset Path Set To: /content/Brain_Cancer raw MRI data/Brain_Cancer


In [9]:
import timm
import timm.data
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

In [10]:
#Loading model (Pretrained)
model = timm.create_model('vit_base_patch16_224', pretrained=True)

#Preprocessing
data_config = timm.data.resolve_model_data_config(model)
transform = timm.data.create_transform(**data_config, is_training=False) #Preprocessing

#Loading Dataset
fullDataset = datasets.ImageFolder(root=dataset, transform=transform)

print("Classes:", fullDataset.classes)

Classes: ['brain_glioma', 'brain_menin', 'brain_tumor']


In [11]:
#Split
total_size = len(fullDataset)
train_size = int(0.7 * total_size)
val_size = int(0.15 * total_size)
test_size = total_size - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(fullDataset, [train_size, val_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

print("Preprocess Images")
print(f"Total Size: {total_size}")
print(f"Train Size: {len(train_dataset)}")
print(f"Val Size: {len(val_dataset)}")
print(f"Test Size: {len(test_dataset)}")

Preprocess Images
Total Size: 6056
Train Size: 4239
Val Size: 908
Test Size: 909


In [12]:
import torch
from torch import nn, optim

#Freeze

for p in model.parameters():
  p.requires_grad = False

model.head = torch.nn.Linear(in_features=model.head.in_features,out_features=len(fullDataset.classes))

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.head.parameters(), lr=1e-4)

In [15]:
import time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

epochs = 3
best_value_acc = 0

start = time.time()

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    running_correct = 0
    running_total = 0

    for imgs, labels in train_loader:
        imgs = imgs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * labels.size(0)
        preds = outputs.argmax(dim=1)
        running_correct += (preds == labels).sum().item()
        running_total += labels.size(0)

    train_loss = running_loss / running_total
    train_acc = running_correct / running_total

    model.eval()
    val_correct = 0
    val_total = 0
    val_loss_sum = 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs = imgs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            val_loss_sum += loss.item() * labels.size(0)
            preds = outputs.argmax(dim=1)
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)

    val_loss = val_loss_sum / val_total
    val_acc = val_correct / val_total
    print(f"Epoch {epoch+1}: train_loss={running_loss:.4f}, val_loss={train_loss:.4f}, val_acc={val_acc:.2f}%")

    if val_acc > best_value_acc:
        best_value_acc = val_acc
        torch.save(model.state_dict(), "best_model.pth")
        print(f"Saved best model (val_acc={best_value_acc:.2f}%)")


print(f"Total Training Time = {(time.time() - start)//(60*60)} hours")

Epoch 1: train_loss=1730.0468, val_loss=0.4081, val_acc=0.86%
Saved best model (val_acc=0.86%)
Epoch 2: train_loss=1494.3445, val_loss=0.3525, val_acc=0.89%
Saved best model (val_acc=0.89%)
Epoch 3: train_loss=1326.5600, val_loss=0.3129, val_acc=0.89%
Saved best model (val_acc=0.89%)
Total Training Time = 0.0 hours


In [18]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
model.load_state_dict(torch.load("best_model.pth"))
model.eval()

all_labels, all_preds = [], []

with torch.no_grad():
  for images, labels in test_loader:
    images, labels = images.to(device), labels.to(device)
    outputs = model(images)
    preds = outputs.argmax(dim=1)
    all_labels.extend(labels.cpu().numpy())
    all_preds.extend(preds.cpu().numpy())

accuracy = accuracy_score(all_labels, all_preds)
precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average="weighted")
cm = confusion_matrix(all_labels, all_preds)
print(f"Accuracy: {accuracy * 100:.2f}%")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-score:  {f1:.4f}")
print("Confusion Matrix:\n", cm)

Accuracy: 90.32%
Precision: 0.9045
Recall:    0.9032
F1-score:  0.9022
Confusion Matrix:
 [[275  17   3]
 [ 21 262  37]
 [  3   7 284]]
